# XGBoost Training for Credit Risk

Trains an XGBoost classifier on the processed credit risk data and evaluates it against the existing MLP baseline.

In [8]:
!pip install xgboost
import xgboost as xgb

In [9]:
import json
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from xgboost import XGBClassifier

PROJECT_ROOT = Path('.').resolve()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

TRAIN_FILE = DATA_DIR / "train.csv"
VAL_FILE = DATA_DIR / "val.csv"
TEST_FILE = DATA_DIR / "test.csv"

MODEL_FILE = MODELS_DIR / "xgboost_model.json"
CALIBRATOR_FILE = MODELS_DIR / "xgboost_calibrator.pkl"
PREDICTIONS_FILE = RESULTS_DIR / "xgboost_predictions.csv"
METRICS_FILE = RESULTS_DIR / "xgboost_metrics.json"

TARGET_COL = "status"
RANDOM_STATE = 42

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_STATE)


In [10]:
# Load data
train_df = pd.read_csv(TRAIN_FILE)
val_df = pd.read_csv(VAL_FILE)
test_df = pd.read_csv(TEST_FILE)

X_train = train_df.drop(columns=[TARGET_COL]).values
y_train = train_df[TARGET_COL].values

X_val = val_df.drop(columns=[TARGET_COL]).values
y_val = val_df[TARGET_COL].values

X_test = test_df.drop(columns=[TARGET_COL]).values
y_test = test_df[TARGET_COL].values

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")
print(f"Class balance (train): {y_train.mean():.1%} default rate")

Training set: (134437, 68)
Validation set: (14867, 68)
Test set: (14867, 68)
Class balance (train): 33.3% default rate


In [11]:
def evaluate_split(name: str, y_true: np.ndarray, proba: np.ndarray) -> dict:
    preds = (proba >= 0.5).astype(int)
    auc_roc = roc_auc_score(y_true, proba)
    auc_pr = average_precision_score(y_true, proba)
    brier = brier_score_loss(y_true, proba)
    positive_rate = y_true.mean()

    print(f"{name} PERFORMANCE (Uncalibrated)")
    print("=" * 80)
    print(f"AUC-ROC:     {auc_roc:.4f}")
    print(f"AUC-PR:      {auc_pr:.4f}")
    print(f"Brier Score: {brier:.4f}")
    print(f"Positives:   {positive_rate:.2%}")
    print("Classification Report:")
    print(classification_report(y_true, preds, digits=4))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, preds))

    return {
        'dataset': name,
        'auc_roc': float(auc_roc),
        'auc_pr': float(auc_pr),
        'brier_score': float(brier)
    }

In [12]:
# Define and train XGBoost model
xgb_params = {
    'n_estimators': 500,
    'learning_rate': 0.05,
    'max_depth': 5,
    'min_child_weight': 1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'gamma': 0.0,
    'reg_lambda': 1.0,
    'reg_alpha': 0.0,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'random_state': RANDOM_STATE,
    'n_jobs': os.cpu_count()
}

model = XGBClassifier(**xgb_params)

print("Training XGBoost model...")
model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,
    early_stopping_rounds=30
)

best_iteration = getattr(model, 'best_iteration', xgb_params['n_estimators'])
print(f"Best iteration: {best_iteration}")

Training XGBoost model...
[0]	validation_0-auc:0.82248


/Users/mac/miniforge3/lib/python3.12/site-packages/xgboost/sklearn.py:835: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


[50]	validation_0-auc:0.86503
[100]	validation_0-auc:0.87251
[150]	validation_0-auc:0.87694
[200]	validation_0-auc:0.87927
[250]	validation_0-auc:0.88045
[300]	validation_0-auc:0.88138
[350]	validation_0-auc:0.88211
[400]	validation_0-auc:0.88267
[450]	validation_0-auc:0.88325
[499]	validation_0-auc:0.88333
Best iteration: 499


In [13]:
# Evaluate on validation and test sets
val_proba = model.predict_proba(X_val)[:, 1]
test_proba = model.predict_proba(X_test)[:, 1]

val_metrics = evaluate_split("Validation", y_val, val_proba)
test_metrics = evaluate_split("Test", y_test, test_proba)

Validation PERFORMANCE (Uncalibrated)
AUC-ROC:     0.8833
AUC-PR:      0.8267
Brier Score: 0.0880
Positives:   24.65%
Classification Report:
              precision    recall  f1-score   support

           0     0.8844    0.9831    0.9311     11203
           1     0.9217    0.6070    0.7319      3664

    accuracy                         0.8904     14867
   macro avg     0.9030    0.7951    0.8315     14867
weighted avg     0.8936    0.8904    0.8820     14867

Confusion Matrix:
[[11014   189]
 [ 1440  2224]]
Test PERFORMANCE (Uncalibrated)
AUC-ROC:     0.8928
AUC-PR:      0.8384
Brier Score: 0.0847
Positives:   24.65%
Classification Report:
              precision    recall  f1-score   support

           0     0.8871    0.9858    0.9339     11203
           1     0.9342    0.6165    0.7428      3664

    accuracy                         0.8948     14867
   macro avg     0.9107    0.8012    0.8384     14867
weighted avg     0.8987    0.8948    0.8868     14867

Confusion Matrix:
[[1

In [14]:
# Calibration with Platt scaling using validation predictions
calibrator = LogisticRegression()
calibrator.fit(val_proba.reshape(-1, 1), y_val)

val_proba_cal = calibrator.predict_proba(val_proba.reshape(-1, 1))[:, 1]
test_proba_cal = calibrator.predict_proba(test_proba.reshape(-1, 1))[:, 1]

val_calibration = {
    'brier_score_calibrated': brier_score_loss(y_val, val_proba_cal),
    'calibration_gap_uncalibrated': abs(y_val.mean() - val_proba.mean()),
    'calibration_gap_calibrated': abs(y_val.mean() - val_proba_cal.mean())
}

test_calibration = {
    'brier_score_calibrated': brier_score_loss(y_test, test_proba_cal),
    'calibration_gap_uncalibrated': abs(y_test.mean() - test_proba.mean()),
    'calibration_gap_calibrated': abs(y_test.mean() - test_proba_cal.mean())
}

print("Calibration analysis complete.")

Calibration analysis complete.


In [16]:
# Save model, calibrator, predictions, and metrics
model.save_model(MODEL_FILE)
joblib.dump(calibrator, CALIBRATOR_FILE)
print(f"Model saved to {MODEL_FILE}")
print(f"Calibrator saved to {CALIBRATOR_FILE}")

predictions_df = pd.DataFrame({
    'true_label': y_test,
    'predicted_probability': test_proba,
    'predicted_probability_calibrated': test_proba_cal,
    'predicted_label': (test_proba >= 0.5).astype(int),
    'predicted_label_calibrated': (test_proba_cal >= 0.5).astype(int)
})
predictions_df.to_csv(PREDICTIONS_FILE, index=False)
print(f"Predictions saved to {PREDICTIONS_FILE}")

metrics = {
    'model': 'XGBoost',
    'hyperparameters': {**xgb_params, 'best_iteration': int(best_iteration)},
    'calibration': 'Platt Scaling on validation set',
    'validation_metrics': {**val_metrics, **val_calibration},
    'test_metrics': {**test_metrics, **test_calibration}
}

with open(METRICS_FILE, 'w') as f:
    json.dump(metrics, f, indent=4)
print(f"Metrics saved to {METRICS_FILE}")

print("" + "=" * 80)
print("XGBoost training complete")
print("=" * 80)

Model saved to /Users/mac/Downloads/MMA/RSM8421/credit-risk-counterfactual/models/xgboost_model.json
Calibrator saved to /Users/mac/Downloads/MMA/RSM8421/credit-risk-counterfactual/models/xgboost_calibrator.pkl
Predictions saved to /Users/mac/Downloads/MMA/RSM8421/credit-risk-counterfactual/results/xgboost_predictions.csv
Metrics saved to /Users/mac/Downloads/MMA/RSM8421/credit-risk-counterfactual/results/xgboost_metrics.json
XGBoost training complete
